# Study 830 — BAB Across Asset Classes — the teardown

The Frazzini-Pedersen rolling betas to the equal-weight multi-asset market, the beta-neutral BAB factor, its Newey-West spread *t* and HAC CAPM alpha, the 1,000-permutation placebo, the two-era robustness cut, the costed levered timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2007-04-11', 'end': '2026-06-30', 'n_assets': 9, 'n_rows': 4836, 'n_days': 4583, 'fingerprint': 'c502a478deee', 'bab_bps': 0.54, 't_nw': 0.31, 't_1s': 0.25, 'sharpe': 0.06, 'alpha_bps': 2.86, 'alpha_t': 1.61, 'realized_beta': -0.83, 'lo_bps': 1.6, 'beta_L': 0.56, 'hi_bps': 3.84, 'beta_H': 1.45, 'placebo_obs': 0.54, 'placebo_mean': 3.515, 'placebo_sd': 1.528, 'placebo_p': 0.984, 'placebo_sigma': 1.95, 'placebo_draws': 1000, 'era1_bps': 3.7, 'era1_tnw': 1.2, 'era1_alpha_t': 1.95, 'era1_n': 2071, 'era2_bps': -2.07, 'era2_tnw': -1.12, 'era2_alpha_t': -0.25, 'era2_n': 2512, 'timer_1_gross': 0.54, 'timer_1_cost': 0.17, 'timer_1_net': 0.37, 'timer_1_t': 0.17, 'timer_5_gross': 0.54, 'timer_5_cost': 0.45, 'timer_5_net': 0.09, 'timer_5_t': 0.04, 'avg_gross': 2.62, 'avg_turnover': 0.071, 'null_mean_t': 0.12, 'null_sd_t': 0.89, 'null_fire': 1, 'planted_t': 3.55, 'planted_alpha_t': 3.92, 'planted_sharpe': 1.06}

## The headline — the beta-neutral BAB factor

Long low-beta (levered by 1/β_L), short high-beta (de-levered by 1/β_H); risk-free ≈ 0.

In [2]:
print(f"BAB return   : {R['bab_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"CAPM alpha   : {R['alpha_bps']:+.2f} bps/day  HAC t = {R['alpha_t']:+.2f}  "
      f"(realized market beta {R['realized_beta']:+.2f})")
print(f"legs         : low-beta {R['lo_bps']:+.2f} bps (beta_L {R['beta_L']:.2f}) vs "
      f"high-beta {R['hi_bps']:+.2f} bps (beta_H {R['beta_H']:.2f})")
print(f"gross Sharpe : {R['sharpe']:.2f} (annualized, before cost)")

BAB return   : +0.54 bps/day  NW(10) t = +0.31  one-sample t = +0.25
CAPM alpha   : +2.86 bps/day  HAC t = +1.61  (realized market beta -0.83)
legs         : low-beta +1.60 bps (beta_L 0.56) vs high-beta +3.84 bps (beta_H 1.45)
gross Sharpe : 0.06 (annualized, before cost)


## Placebo — column-permute the asset returns (1,000 permutations)

Keep the leverage structure; shuffle *which* asset each leg holds, breaking the beta→return link. Because the levered book is mechanically ~net-long the average asset, the permutation null centres **above** zero; the question is whether the *actual* beta mapping beats a random relabelling.

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> two-sided p = {R['placebo_p']:.4f} "
      f"({R['placebo_sigma']:.2f} sigma)")
print('  -> the beta sort adds NOTHING beyond the mechanical net-long tilt '
      '(observed sits at the low end of the permutation cloud)')

observed +0.54 bps vs placebo mean +3.515 (sd 1.528) -> two-sided p = 0.9840 (1.95 sigma)
  -> the beta sort adds NOTHING beyond the mechanical net-long tilt (observed sits at the low end of the permutation cloud)


## Robustness — two eras (split 2016-07-01)

In [4]:
print(f"2007-2016 (n={R['era1_n']}): {R['era1_bps']:+.2f} bps  NW t = {R['era1_tnw']:+.2f}  alpha t = {R['era1_alpha_t']:+.2f}")
print(f"2016-2026 (n={R['era2_n']}): {R['era2_bps']:+.2f} bps  NW t = {R['era2_tnw']:+.2f}  alpha t = {R['era2_alpha_t']:+.2f}")
print('  -> weak-only pre-2016, sign-flips after: not a stable edge')

2007-2016 (n=2071): +3.70 bps  NW t = +1.20  alpha t = +1.95
2016-2026 (n=2512): -2.07 bps  NW t = -1.12  alpha t = -0.25
  -> weak-only pre-2016, sign-flips after: not a stable edge


## The timer — can you get paid for it?

Realized daily turnover × one-way cost × NAV on the ~2.6×-gross levered book; short leg pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")
print(f"  avg gross leverage {R['avg_gross']:.2f}x, turnover {R['avg_turnover']:.3f}/day")

 1 bp one-way: gross +0.54 -> net +0.37 bps/day (cost 0.17/day, t=+0.17)
5 bps one-way: gross +0.54 -> net +0.09 bps/day (cost 0.45/day, t=+0.04)
  avg gross leverage 2.62x, turnover 0.071/day


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null (CAPM holds) and must recover a planted flat-SML premium.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from bab_multiasset import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_series(edge=0.0, seed=830+s, n_days=2500))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_series(edge=0.0006, seed=830, n_days=2500))
print(f"planted (edge=0.0006): NW t = {planted['t_nw']:+.2f}, alpha t = {planted['alpha_t']:+.2f}, Sharpe {planted['sharpe']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.08 (sd 0.59), |t|>=2 in 0/8


planted (edge=0.0006): NW t = +3.55, alpha t = +3.92, Sharpe +1.06


## Verdict

- **Signal — None.** The multi-asset BAB factor earns **+0.54 bps/day** (NW *t* = **+0.31**); as a CAPM alpha only **+2.86 bps/day** (HAC *t* = +1.61) — neither clears |t| ≥ 2. The book's realized market beta is **-0.83** (the 'low-beta' leg is Treasuries + gold, so this is a disguised long-duration bet, not a clean SML arbitrage); it worked weakly pre-2016 (alpha *t* = +1.95) and reversed after (-0.25); and the 1,000-permutation placebo shows the beta sort adds nothing beyond the mechanical net-long tilt (observed +0.54 vs cloud mean +3.52, two-sided p = 0.98). The 20-seed synthetic control recovers a *planted* flat-SML premium cleanly (*t* = +3.55, fires 1/20 nulls ≈ the nominal 5%), so the machinery is sound — the cross-asset SML simply is not flat enough.
- **Tradability — Mirage.** The factor is flat gross; net of realized turnover on a 2.6×-gross levered book it is **+0.37 bps/day** at 1 bp (*t* = +0.17) and **+0.09** at 5 bps — indistinguishable from zero at any cost.